# Transformer Training Notebook
This notebook sets up and trains a FullTransformer model using the TransformerTrainer class on the Toxic Comment Classification dataset.

In [15]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
import sys, os

# If running in Colab, mount drive

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive/OMSCS/DeepLearning/main_repo'
    os.chdir(BASE_PATH)
    sys.path.append(BASE_PATH)
    print(f'Running in Colab, path set to {BASE_PATH}')
except ImportError:
    print('Running locally')

Running locally


In [17]:
# Dataset setup

import re, random
import nltk, numpy as np, pandas as pd, torch
from nltk.tokenize import word_tokenize
from torch.utils.data import Dataset, DataLoader
from collections import Counter

# Download punkt if needed
nltk.download('punkt')

# Data utility functions
from dataset.data_cleaning import clean_text, preprocess_and_tokenize, load_glove_embeddings, ToxicDataset

def seed_torch(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_torch(42)

[nltk_data] Downloading package punkt to /Users/soultanis/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
# Transformer configuration

class Config:
    EMBEDDING_DIM = 100
    MAX_LEN = 100
    BATCH_SIZE = 32
    NUM_EPOCHS = 20
    LR = 1e-4
    GAMMA = 0.95
    NUM_HEADS = 5
    NUM_ENCODING_LAYERS = 4
    DROPOUT = 0.1
    GLOVE_PATH = 'dataset/word_embeddings/glove.6B/glove.6B.100d.txt'
    DATA_PATH = 'dataset/.cache/kagglehub/competitions/jigsaw-toxic-comment-classification-challenge/train.csv.zip'
    EMBEDDING_PATH = f"dataset/word_embeddings/glove.6B/glove.6B.100d.txt"
    LABELS = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']
    SAVE_TEST_CSV_PATH = "test_file.csv"
    SEED = 42
    LOSS_TYPE = 'BCE' # or 'FOCAL'

In [19]:
import torch.nn as nn

class CBFocalLoss(nn.Module):
    def __init__(self, samples_per_cls, beta=0.9999, gamma=2.0):
        super(CBFocalLoss, self).__init__()
        self.beta = beta
        self.gamma = gamma

        effective_num = 1.0 - torch.pow(self.beta, samples_per_cls)
        weights = (1.0 - self.beta) / (effective_num + 1e-8)
        weights = weights / weights.sum() * len(samples_per_cls)  # Normalize

        self.class_weights = weights.to(torch.float32)

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = torch.clamp(probs, 1e-4, 1 - 1e-4)

        # 👇 Move weights to same device as logits
        weights = self.class_weights.to(logits.device).unsqueeze(0)

        focal_weight = torch.pow(1.0 - probs * targets - (1 - probs) * (1 - targets), self.gamma)

        bce = - (targets * torch.log(probs) + (1 - targets) * torch.log(1 - probs))
        loss = weights * focal_weight * bce

        return loss.mean()

In [20]:
# Load and preprocess data

cfg = Config()
df = pd.read_csv(cfg.DATA_PATH)
df, vocab = preprocess_and_tokenize(df, cfg.MAX_LEN)
embedding_matrix = load_glove_embeddings(cfg.GLOVE_PATH, vocab, cfg.EMBEDDING_DIM)
samples_per_cls = torch.tensor([df[col].sum() for col in cfg.LABELS], dtype=torch.float32)

print('Vocab size:', len(vocab))
print('Class counts:', samples_per_cls)

Vocab size: 245832
Class counts: tensor([15294.,  1595.,  8449.,   478.,  7877.,  1405.])


In [21]:
# Prepare DataLoaders
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    df['input_ids'].tolist(), df[cfg.LABELS].values.tolist(), test_size=0.2, random_state=cfg.SEED
)

train_ds = ToxicDataset(X_train, y_train)
val_ds = ToxicDataset(X_val, y_val)
train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE)

In [ ]:
# Initialize model and trainer
import torch.nn as nn
from models.transformer.model import TransformerClassifier
from models.transformer.trainer import TransformerTrainer

if torch.cuda.is_available():
    device = torch.device("cuda")
# elif torch.backends.mps.is_available():
#     device = torch.device("mps")
else:
    device = torch.device("cpu")

glove_tensor = torch.from_numpy(embedding_matrix).float()
num_classes = len(cfg.LABELS)

model = TransformerClassifier(
    src_glove_weights=glove_tensor,
    pad_idx=vocab.get('<PAD>', 0),
    num_classes=num_classes,
    d_model=cfg.EMBEDDING_DIM,
    nhead=cfg.NUM_HEADS,
    num_encoder_layers=cfg.NUM_ENCODING_LAYERS,
    dim_feedforward=2048,
    dropout=cfg.DROPOUT,
    max_len=cfg.MAX_LEN,
).to(device)

ignore_index = vocab.get('<PAD>', 0)

samples_per_cls = torch.tensor([df[col].sum() for col in cfg.LABELS], dtype=torch.float32).to(device)

if cfg.LOSS_TYPE == 'BCE':
    criterion = nn.BCEWithLogitsLoss()
else: 
    criterion = CBFocalLoss(samples_per_cls=samples_per_cls)


# Instantiate the updated trainer
trainer = TransformerTrainer(
    model=model,
    pad_idx=ignore_index,
    criterion=criterion,
    lr=cfg.LR,
    clip_grad_norm=1.0,
    device=device
)

In [25]:
print(device)

# Training loop for the model
trainer.train(train_loader, val_loader, 1)


cpu
Starting training...
Epoch 1: Train Loss = 0.1167, Val Loss = 0.0807, LR = 0.000010


In [ ]:
# Save trained transformer
torch.save(model.state_dict(), 'transformer_toxic_comments.pth')

In [ ]:
# Perform inference on test data

from nltk.tokenize import word_tokenize
from dataset.data_cleaning import clean_text


def prepare_test_data(df, vocab, max_len):
    df['comment_text'] = df['comment_text'].apply(clean_text)
    df['tokens'] = df['comment_text'].apply(word_tokenize)

    def encode(tokens):
        return [vocab.get(tok, vocab['<UNK>']) for tok in tokens[:max_len]]

    def pad(seq):
        return seq + [0] * (max_len - len(seq))

    df['input_ids'] = df['tokens'].apply(lambda x: pad(encode(x)))
    return df


def run_inference(model, test_loader, device):
    model.eval()
    predictions = []

    with torch.no_grad():
        for inputs, _ in test_loader:  # labels are dummy here
            inputs = inputs.to(device)
            logits = model(inputs)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs > 0.5).astype(int)
            predictions.extend(preds)

    return np.array(predictions)


# Load test data
test_df = pd.read_csv("dataset/.cache/kagglehub/competitions/jigsaw-toxic-comment-classification-challenge/test.csv.zip")
test_labels_df = pd.read_csv("dataset/.cache/kagglehub/competitions/jigsaw-toxic-comment-classification-challenge/test_labels.csv.zip")  # Optional

# Prepare
test_df = prepare_test_data(test_df, vocab, cfg.MAX_LEN)
dummy_labels = [[0]*len(cfg.LABELS)] * len(test_df)  # dummy labels to match dataset format
test_ds = ToxicDataset(test_df['input_ids'].tolist(), dummy_labels)
test_loader = DataLoader(test_ds, batch_size=cfg.BATCH_SIZE)

# Load trained model
checkpoint = torch.load('transformer_toxic_comments.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Run inference
preds = run_inference(model, test_loader, device)

# Save predictions
output_df = test_df.copy()
output_df[cfg.LABELS] = preds
output_df.to_csv('test_predictions_transformer_with_bifocal_loss.csv', index=False)